# Basado en [A Survey on Deep Neural Networks in Collaborative Filtering Recommendation Systems](https://arxiv.org/abs/2412.01378)



# Load data

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import polars as pl
from collaborative import diagnose_cold_start, print_cold_start_report
from collaborative import ColdStartStatus
import os

ModuleNotFoundError: No module named 'torch'

In [ ]:
train_data = pl.read_csv("./data/collaborative_filtering/train.csv")
test_data = pl.read_csv("./data/collaborative_filtering/test.csv")
train_data.head()

In [ ]:
train_shuffled = train_data.sample(fraction=1.0, shuffle=True, seed=42)

split_idx = int(0.8 * train_shuffled.height)

train_subset = train_shuffled[:split_idx]
eval_set = train_shuffled[split_idx:]

def attach_status_polars(df: pl.DataFrame, status_map: dict[int, ColdStartStatus]) -> pl.DataFrame:
    # Convertimos el Enum a string (.name) para que Polars trabaje con tipo de dato String nativo
    status_list = [status_map[i].name for i in range(df.height)]
    
    return df.with_columns(
        pl.Series("status", status_list)
    )

# 1. Seleccionamos columnas originales
train_df = train_subset.select(["user", "item", "rating"])
eval_df  = eval_set.select(["user", "item", "rating"])

# 2. Preparamos el array de train (user_id, item_id, rating) -> índices 0 y 1. Esto es correcto.
train_np = train_df.to_numpy()
eval_np = eval_df.to_numpy()

# Añadimos una columna contador porque lo pide la fx
eval_np = (
    eval_df
    .with_row_index("id")  # crea columna id = índice
    .select(["id", "user", "item"])
    .to_numpy()
)


# 4. Diagnóstico pasándole los arrays con los índices correctos
status_map_eval, summary_eval = diagnose_cold_start(train_np, eval_np)


# 5. Añadir columna status 

eval_df = attach_status_polars(eval_df, status_map_eval)
# train_df = attach_status_polars(train_df, status_map_train)


def split_by_status(df: pl.DataFrame):
    return {
        "ok": df.filter(pl.col("status") == ColdStartStatus.OK.name),
        "cold_user": df.filter(pl.col("status") == ColdStartStatus.COLD_USER.name),
        "unknown_item": df.filter(pl.col("status") == ColdStartStatus.UNKNOWN_ITEM.name),
        "both": df.filter(pl.col("status") == ColdStartStatus.BOTH.name),
    }

eval_splits = split_by_status(eval_df)
# train_splits = split_by_status(train_df)

eval_ok = eval_splits["ok"]
eval_cold_user = eval_splits["cold_user"]
eval_unknown_item = eval_splits["unknown_item"]
eval_both = eval_splits["both"]

# train_ok = train_splits["ok"]
# train_cold_user = train_splits["cold_user"]
# train_unknown_item = train_splits["unknown_item"]
# train_both = train_splits["both"]


print("Eval summary (from function):", summary_eval)
print("Eval counts (from df):")
print(print_cold_start_report(summary_eval))

# print("Train summary (from function):", summary_train)
# print("Train counts (from df):")
# print(count_by_status(train_df))

In [ ]:
import joblib
from sklearn.preprocessing import LabelEncoder

# Definir el formato de los datos
# Para poder trabajar con deep learning tenemos que pasar los IDs de 
# usuarios e items a un vector numerico continuo
folder_path = "./ml_assets"
os.makedirs(folder_path, exist_ok=True)
user_enc = LabelEncoder()
item_enc = LabelEncoder()

train_df = train_df.with_columns([
    pl.Series("user_idx", user_enc.fit_transform(train_df["user"].to_numpy())),
    pl.Series("item_idx", item_enc.fit_transform(train_df["item"].to_numpy()))
])
eval_ok = eval_ok.with_columns([
    pl.Series("user_idx", user_enc.transform(eval_ok["user"].to_numpy())),
    pl.Series("item_idx", item_enc.transform(eval_ok["item"].to_numpy()))
])

N_USERS = len(user_enc.classes_)
N_ITEMS = len(item_enc.classes_)

joblib.dump(user_enc, os.path.join(folder_path, 'user_encoder.joblib'))
joblib.dump(item_enc, os.path.join(folder_path, 'item_encoder.joblib'))


# Modelo de Deep Learning


In [3]:
class NeuMF(nn.Module):
    def __init__(self, num_users, num_items, latent_dim=8, layers=[64, 32, 16, 8]):
        super(NeuMF, self).__init__()
        
        # --- Rama GMF ---
        # Embeddings para la Factorización de Matrices Generalizada
        self.embedding_user_GMF = nn.Embedding(num_users, latent_dim)
        self.embedding_item_GMF = nn.Embedding(num_items, latent_dim)
        
        # --- Rama MLP ---
        # Embeddings para el Perceptrón Multicapa
        self.embedding_user_MLP = nn.Embedding(num_users, layers[0] // 2)
        self.embedding_item_MLP = nn.Embedding(num_items, layers[0] // 2)
        
        # Capas ocultas del MLP
        mlp_modules = []
        for i in range(len(layers) - 1):
            mlp_modules.append(nn.Linear(layers[i], layers[i+1]))
            mlp_modules.append(nn.ReLU())
        self.mlp_network = nn.Sequential(*mlp_modules)
        
        # --- Capa de Predicción Final ---
        # Combina la salida de GMF (dimensión latent_dim) y MLP (dimensión layers[-1])
        predict_size = latent_dim + layers[-1]
        self.prediction_layer = nn.Linear(predict_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, user_indices, item_indices):
        # Procesamiento GMF
        user_gmf = self.embedding_user_GMF(user_indices)
        item_gmf = self.embedding_item_GMF(item_indices)
        mf_vector = torch.mul(user_gmf, item_gmf) # Producto elemental
        
        # Procesamiento MLP
        user_mlp = self.embedding_user_MLP(user_indices)
        item_mlp = self.embedding_item_MLP(item_indices)
        # Concatenación de vectores
        mlp_vector = torch.cat([user_mlp, item_mlp], dim=-1)
        mlp_vector = self.mlp_network(mlp_vector)
        
        # Fusión y Predicción 
        combined_vector = torch.cat([mf_vector, mlp_vector], dim=-1)
        prediction = self.prediction_layer(combined_vector)
        
        return self.sigmoid(prediction).view(-1)


NameError: name 'nn' is not defined

# Torch dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader

class NCFDataset(Dataset):
    def __init__(self, df: pl.DataFrame):
        # Convertimos columnas de Polars a Tensores de PyTorch
        self.users = torch.tensor(df["user_idx"].to_numpy(), dtype=torch.long)
        self.items = torch.tensor(df["item_idx"].to_numpy(), dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

# Crear instancias para entrenamiento y validación
train_dataset = NCFDataset(train_df)
eval_dataset = NCFDataset(eval_ok)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=256, shuffle=False)

# Entrenamiento

In [ ]:
# Parámetros basados en tus encoders
model = NeuMF(num_users=N_USERS, num_items=N_ITEMS, latent_dim=16)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

# Bucle de entrenamiento simplificado
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on {device}")
model.to(device)

import matplotlib.pyplot as plt

train_losses = []

model.to(device)
for epoch in range(10):
    model.train()
    epoch_loss = 0
    for batch_user, batch_item, batch_rating in train_loader:
        batch_user, batch_item, batch_rating = batch_user.to(device), batch_item.to(device), batch_rating.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_user, batch_item)
        loss = criterion(predictions, batch_rating)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

# --- Generación del Plot ---
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss', color='royalblue', marker='o')
plt.title('Progreso del Entrenamiento (NeuMF)')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

NameError: name 'NeuMF' is not defined